# 01 - Entender el dataset IMPROVE/CSA

Este notebook tiene como objetivo explicar de forma sencilla la estructura del dataset utilizado en el TFG.

El trabajo parte de una idea básica:

**expresión génica de la línea celular + representación molecular del fármaco → respuesta AUC**

Por tanto, antes de entrenar modelos como DeepTTC o Random Forest, es importante entender qué contiene cada fichero, cómo se relacionan entre sí y qué representa cada fila del problema.

En este notebook se revisan los tres ficheros principales:

- `response.tsv`: contiene los pares célula-fármaco y la respuesta farmacológica.
- `drug_SMILES.tsv`: contiene la representación molecular de los fármacos.
- `cancer_gene_expression.tsv`: contiene la expresión génica de las líneas celulares.

También se revisan los splits y la distribución de la variable objetivo AUC.


## 1. Carga de librerías y definición de rutas

Primero se cargan las librerías necesarias y se definen las rutas principales del proyecto.

El notebook está pensado para ejecutarse desde la carpeta `notebooks_tfg/`. Por eso se toma como raíz del proyecto la carpeta anterior (`..`). Si se ejecuta desde otra ubicación, basta con modificar la variable `ROOT`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Si el notebook está dentro de notebooks_tfg/, la raíz del proyecto es la carpeta anterior.
ROOT = Path("..").resolve()

response_path = ROOT / "csa_data/raw_data/y_data/response.tsv"
smiles_path = ROOT / "csa_data/raw_data/x_data/drug_SMILES.tsv"
gene_path = ROOT / "csa_data/raw_data/x_data/cancer_gene_expression.tsv"
splits_dir = ROOT / "csa_data/raw_data/splits"

print("ROOT:", ROOT)
print("response_path:", response_path)
print("smiles_path:", smiles_path)
print("gene_path:", gene_path)
print("splits_dir:", splits_dir)


## 2. Carga de los ficheros principales

En este TFG se utilizan tres fuentes de información:

1. **Respuesta farmacológica**: indica cómo responde una célula a un fármaco.
2. **SMILES del fármaco**: representa la estructura química del fármaco como texto.
3. **Expresión génica**: representa el estado molecular de cada línea celular.

Estas tres fuentes se combinan posteriormente para construir los ejemplos de entrenamiento.


In [ ]:
response = pd.read_csv(response_path, sep="\t")
smiles = pd.read_csv(smiles_path, sep="\t")
genes = pd.read_csv(gene_path, sep="\t")

print("response:", response.shape)
print("smiles:", smiles.shape)
print("genes:", genes.shape)


## 3. Primer vistazo a los datos

A continuación se muestran las primeras filas de cada fichero. Esto permite identificar rápidamente las columnas principales y el tipo de información almacenada.


In [ ]:
response.head()

In [ ]:
smiles.head()

In [ ]:
genes.head()

## 4. Explicación de cada fichero

### `response.tsv`

Cada fila representa una combinación entre una línea celular y un fármaco. Las columnas más importantes para este trabajo son:

- `improve_sample_id`: identificador de la línea celular.
- `improve_chem_id`: identificador del fármaco.
- `auc`: respuesta farmacológica que se quiere predecir.

### `drug_SMILES.tsv`

Asocia cada fármaco con su representación molecular en formato SMILES.

Un SMILES es una forma textual de representar una molécula. En lugar de utilizar una imagen química, la estructura del fármaco se codifica como una cadena de caracteres.

### `cancer_gene_expression.tsv`

Contiene la matriz de expresión génica de las líneas celulares. Cada fila corresponde a una línea celular y cada columna representa una variable génica.


## 5. Columnas disponibles

Se listan las columnas principales de cada fichero para comprobar que los identificadores y variables esperadas están presentes.


In [ ]:
print("Columnas response:")
print(response.columns.tolist())

print("\nColumnas smiles:")
print(smiles.columns.tolist())

print("\nPrimeras columnas genes:")
print(genes.columns.tolist()[:20])

print("\nNúmero total de columnas en genes:", len(genes.columns))


## 6. Comprobación de valores perdidos

Antes de entrenar cualquier modelo, conviene revisar si existen valores nulos. La presencia de nulos puede afectar al preprocesamiento, al entrenamiento o al cálculo de métricas.


In [ ]:
print("NaNs response:", response.isna().sum().sum())
print("NaNs smiles:", smiles.isna().sum().sum())
print("NaNs genes:", genes.isna().sum().sum())


## 7. Identificación de columnas clave

Para que el análisis sea más robusto, se detectan automáticamente las columnas relacionadas con:

- identificador de célula;
- identificador de fármaco;
- variable objetivo AUC;
- representación SMILES.


In [ ]:
sample_cols = [c for c in response.columns if "sample" in c.lower() or "cell" in c.lower()]
drug_cols = [c for c in response.columns if "chem" in c.lower() or "drug" in c.lower()]
auc_cols = [c for c in response.columns if "auc" in c.lower()]
smiles_cols = [c for c in smiles.columns if "smiles" in c.lower()]
smiles_drug_cols = [c for c in smiles.columns if "chem" in c.lower() or "drug" in c.lower()]

print("Columnas candidatas célula:", sample_cols)
print("Columnas candidatas fármaco en response:", drug_cols)
print("Columnas candidatas AUC:", auc_cols)
print("Columnas candidatas SMILES:", smiles_cols)
print("Columnas candidatas fármaco en smiles:", smiles_drug_cols)

sample_col = sample_cols[0]
drug_col = drug_cols[0]
auc_col = auc_cols[0]
smiles_col = smiles_cols[0]
smiles_drug_col = smiles_drug_cols[0]

print("\nUsando:")
print("sample_col:", sample_col)
print("drug_col:", drug_col)
print("auc_col:", auc_col)
print("smiles_col:", smiles_col)
print("smiles_drug_col:", smiles_drug_col)


## 8. Distribución de AUC

AUC es la variable objetivo del problema. El modelo intenta predecir este valor para cada par célula-fármaco.

La distribución de AUC es importante porque ayuda a entender el rango de valores, la concentración de ejemplos y posibles desequilibrios en la respuesta.


In [ ]:
print(response[auc_col].describe())

plt.figure()
plt.hist(response[auc_col], bins=30)
plt.xlabel("AUC")
plt.ylabel("Frecuencia")
plt.title("Distribución global de AUC")
plt.tight_layout()
plt.show()


## 9. Número de líneas celulares, fármacos y pares

Cada ejemplo del problema es un par célula-fármaco. Por eso se revisa cuántas células, fármacos y combinaciones existen en el dataset.


In [ ]:
n_cells = response[sample_col].nunique()
n_drugs = response[drug_col].nunique()
n_pairs = len(response)

print("Número de líneas celulares:", n_cells)
print("Número de fármacos:", n_drugs)
print("Número de pares célula-fármaco:", n_pairs)


## 10. Relación entre respuesta y SMILES

El fichero `response.tsv` contiene el identificador del fármaco, pero no directamente el SMILES. Por eso se puede unir con `drug_SMILES.tsv` usando el identificador de fármaco.

Esta unión permite ver, para cada par célula-fármaco, cuál es la estructura molecular textual del fármaco.


In [ ]:
response_smiles = response[[sample_col, drug_col, auc_col]].merge(
    smiles[[smiles_drug_col, smiles_col]],
    left_on=drug_col,
    right_on=smiles_drug_col,
    how="left"
)

print("response + smiles:", response_smiles.shape)
print("SMILES no encontrados:", response_smiles[smiles_col].isna().sum())

response_smiles.head()


## 11. Relación entre respuesta y expresión génica

La expresión génica se une con las respuestas usando el identificador de línea celular. Esta comprobación permite asegurar que las células presentes en `response.tsv` tienen su vector de expresión génica disponible.


In [ ]:
gene_sample_col = genes.columns[0]

response_genes = response[[sample_col, drug_col, auc_col]].merge(
    genes[[gene_sample_col]],
    left_on=sample_col,
    right_on=gene_sample_col,
    how="left"
)

print("Columna identificadora en genes:", gene_sample_col)
print("response + genes:", response_genes.shape)
print("Células sin expresión génica:", response_genes[gene_sample_col].isna().sum())


## 12. Ejemplo de muestra de entrenamiento

Una muestra de entrenamiento se construye combinando:

1. Una línea celular.
2. El vector de expresión génica de esa línea celular.
3. Un fármaco.
4. El SMILES del fármaco.
5. El valor AUC observado para ese par.

Por tanto:

**entrada del modelo = expresión génica + representación del fármaco**

**salida del modelo = AUC**


In [ ]:
example = response_smiles.iloc[0]

print("Ejemplo de par célula-fármaco")
print("-----------------------------")
print("Línea celular:", example[sample_col])
print("Fármaco:", example[drug_col])
print("SMILES:", example[smiles_col])
print("AUC:", example[auc_col])


## 13. Revisión de splits

Los splits definen qué ejemplos se usan para entrenamiento, validación y test.

En el split normal, los pares célula-fármaco de train, validación y test son diferentes. Sin embargo, las mismas células o los mismos fármacos pueden aparecer en varios subconjuntos. Por este motivo, en el TFG también se estudiaron escenarios más exigentes:

- **cell-out**: células de test no vistas durante entrenamiento.
- **drug-out**: fármacos de test no vistos durante entrenamiento.


In [ ]:
split_files = sorted(splits_dir.glob("CCLE_split_0_*.txt"))

for f in split_files:
    with open(f, "r") as fh:
        lines = [line.strip() for line in fh if line.strip()]
    print(f.name, "->", len(lines), "líneas")


## 14. Conclusiones del análisis exploratorio

Este análisis inicial permite identificar los tres componentes principales del problema:

- la respuesta farmacológica (`response.tsv`);
- la representación molecular del fármaco (`drug_SMILES.tsv`);
- la expresión génica de la línea celular (`cancer_gene_expression.tsv`).

Cada muestra del problema se construye a partir de un par célula-fármaco. La entrada está formada por la expresión génica de la célula y la representación del fármaco, y la salida es el valor de AUC.

También se observa que el tipo de partición es fundamental para interpretar los resultados. El split normal evalúa pares nuevos, pero no necesariamente fármacos o células completamente nuevos. Por ello, los escenarios `cell-out` y `drug-out` son importantes para estudiar la generalización real del modelo.
